# Foundation Model - TabPFN zero-shot #

In [ ]:
from pathlib import Path
import time
import json
from tqdm.auto import tqdm
import gc

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
import sys
scripts_dir = (Path.cwd() / ".." / "scripts").resolve()
sys.path.append(str(scripts_dir))
from config import final_data_path, hf_token

In [2]:
!pip install tabpfn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 21.6 MB/s eta 0:00:00


In [ ]:
final_data_path = Path(final_data_path)
features_dir = final_data_path
results_dir = final_data_path / "modeling_results" / "fm"
results_dir.mkdir(parents=True, exist_ok=True)

layer_files = [
    "layer_a_bts",
    "layer_b_bts_meteostat",
    "layer_c_bts_gdelt",
    "layer_d_bts_meteostat_gdelt",
]

target_col = "dep_del15"
non_feature_cols = ["flight_id", "year", target_col]

datasets = {}
for name in layer_files:
    key = f"{name}_subsampled"
    df = pd.read_parquet(features_dir / f"{key}_features.parquet")
    datasets[key] = df
    print(f"{key}: {df.shape}")

layer_a_bts_subsampled: (6966559, 28)
layer_b_bts_meteostat_subsampled: (6966559, 32)
layer_c_bts_gdelt_subsampled: (6966559, 43)
layer_d_bts_meteostat_gdelt_subsampled: (6966559, 47)


In [4]:
def get_train_test(key: str):
    """Split one dataset by year and drop non-feature columns from X."""
    df = datasets[key]
    train = df[df["year"] == 2024]
    test = df[df["year"] == 2025]

    feature_cols = [c for c in df.columns if c not in non_feature_cols]
    X_train, y_train = train[feature_cols], train[target_col]
    X_test, y_test = test[feature_cols], test[target_col]
    return X_train, y_train, X_test, y_test

In [5]:
def get_train_val_test(key: str, val_size=0.15, random_state=42):
    df = datasets[key]
    train_full = df[df["year"] == 2024]
    test = df[df["year"] == 2025]

    feature_cols = [c for c in df.columns if c not in non_feature_cols]
    train, val = train_test_split(
        train_full, test_size=val_size, stratify=train_full[target_col], random_state=random_state
    )

    X_train, y_train = train[feature_cols], train[target_col]
    X_val, y_val = val[feature_cols], val[target_col]
    X_test, y_test = test[feature_cols], test[target_col]
    return X_train, y_train, X_val, y_val, X_test, y_test

In [6]:
beta=1.2

def tune_threshold(y_true, oob_proba, beta=beta):
    precision, recall, thresholds = precision_recall_curve(y_true, oob_proba)
    fbeta_scores = (
        (1 + beta**2) * precision * recall
        / (beta**2 * precision + recall + 1e-12)
    )
    best_idx = np.nanargmax(fbeta_scores[:-1])
    return thresholds[best_idx]

In [7]:
def predict_proba_chunked(clf, X_test, batch_size=50_000, desc="predicting batches"):
    n = len(X_test)
    proba = np.empty(n, dtype=np.float64)

    for start in tqdm(range(0, n, batch_size), desc=desc):
        end = min(start + batch_size, n)
        proba[start:end] = clf.predict_proba(X_test.iloc[start:end])[:, 1]

    return proba

In [ ]:
import os
os.environ["TABPFN_MODEL_VERSION"] = "v3.5"
os.environ["TABPFN_TOKEN"] = hf_token

In [ ]:
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

X_warm, y_warm, _, _ = get_train_test("layer_a_bts_subsampled")
clf = TabPFNClassifier()
clf.fit(X_warm.iloc[:100], y_warm.iloc[:100])
print("license accepted, checkpoint downloaded")

tabpfn-v3.5-20260909.safetensors:   0%|          | 0.00/876M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

license accepted, checkpoint downloaded


Testing best parameters for n_estimators and subsample_samples to balance performance and GPU time.

In [ ]:
search_layer = "layer_d_bts_meteostat_gdelt_subsampled"
search_train_sample_size = 100_000
search_probe_size = 30_000
search_batchsize = 20_000

n_estimators_grid = [1, 2, 4]
subsample_samples_grid = [10_000, 20_000, 40_000]

X_train_s, y_train_s, X_val_s, y_val_s, X_test_s, y_test_s = get_train_val_test(search_layer)
n_full_test = len(X_test_s)

if len(X_train_s) > search_train_sample_size:
    X_train_s, _, y_train_s, _ = train_test_split(
        X_train_s, y_train_s,
        train_size=search_train_sample_size,
        stratify=y_train_s, random_state=42,
    )

# small, stratified probe slice of the real test set for scoring + timing
_, X_probe, _, y_probe = train_test_split(
    X_test_s, y_test_s,
    test_size=search_probe_size,
    stratify=y_test_s, random_state=42,
)

search_results = []

for n_est in n_estimators_grid:
    for sub_samples in subsample_samples_grid:
        try:
            t0 = time.time()
            clf = TabPFNClassifier.create_default_for_version(
                ModelVersion.V3_5,
                device="cuda",
                n_estimators=n_est,
                inference_config={"SUBSAMPLE_SAMPLES": sub_samples},
                random_state=42,
            )
            clf.fit(X_train_s, y_train_s)
            fit_time = time.time() - t0

            t1 = time.time()
            probe_proba = predict_proba_chunked(
                clf, X_probe, batch_size=search_batchsize,
                desc=f"probe n_est={n_est} sub={sub_samples}",
            )
            predict_time_probe = time.time() - t1

            # extrapolate probe predict time to the full test set for this layer
            predict_time_full_est = predict_time_probe * (n_full_test / len(X_probe))
            total_time_full_est = fit_time + predict_time_full_est

            auc = roc_auc_score(y_probe, probe_proba)
            f1_default = f1_score(y_probe, (probe_proba >= 0.5).astype(int))

            search_results.append({
                "n_estimators": n_est,
                "subsample_samples": sub_samples,
                "probe_auc_roc": auc,
                "probe_f1_at_0.5": f1_default,
                "fit_seconds": fit_time,
                "predict_seconds_probe": predict_time_probe,
                "est_predict_seconds_full_layer": predict_time_full_est,
                "est_total_seconds_full_layer": total_time_full_est,
            })

            print(f"n_estimators={n_est}, subsample_samples={sub_samples} -> "
                  f"AUC={auc:.4f}, F1@0.5={f1_default:.4f}, "
                  f"est. full-layer time={total_time_full_est/60:.1f} min")

        except RuntimeError as e:
            print(f"n_estimators={n_est}, subsample_samples={sub_samples} FAILED: {e}")
        finally:
            del clf
            gc.collect()
            torch.cuda.empty_cache()

search_df = pd.DataFrame(search_results).sort_values("probe_auc_roc", ascending=False)
search_df.to_csv(results_dir / "param_search_fm.csv", index=False)
search_df

probe n_est=1 sub=10000:   0%|          | 0/2 [00:00<?, ?it/s]

n_estimators=1, subsample_samples=10000 -> AUC=0.6776, F1@0.5=0.0623, est. full-layer time=31.7 min


probe n_est=1 sub=20000:   0%|          | 0/2 [00:00<?, ?it/s]

n_estimators=1, subsample_samples=20000 -> AUC=0.6827, F1@0.5=0.0930, est. full-layer time=59.1 min


probe n_est=1 sub=40000:   0%|          | 0/2 [00:00<?, ?it/s]

...

probe n_est=4 sub=20000:   0%|          | 0/2 [00:00<?, ?it/s]

n_estimators=4, subsample_samples=20000 -> AUC=0.6832, F1@0.5=0.0755, est. full-layer time=253.3 min


probe n_est=4 sub=40000:   0%|          | 0/2 [00:00<?, ?it/s]

n_estimators=4, subsample_samples=40000 -> AUC=0.6817, F1@0.5=0.0902, est. full-layer time=647.7 min


,n_estimators,subsample_samples,probe_auc_roc,probe_f1_at_0.5,fit_seconds,predict_seconds_probe,est_predict_seconds_full_layer,est_total_seconds_full_layer
7,4,20000,0.683232,0.075536,0.785457,76.400780,15194.991996,15195.777452
6,4,10000,0.683001,0.043567,0.723585,36.481464,7255.626941,7256.350526
1,1,20000,0.682742,0.092991,0.877934,17.825693,3545.268279,3546.146212
5,2,40000,0.682208,0.084479,0.739364,97.851121,19461.149640,19461.889004
3,2,10000,0.682126,0.050662,0.668601,18.630501,3705.332834,3706.001435
4,2,20000,0.682078,0.077500,0.677145,37.900488,7537.850001,7538.527145
8,4,40000,0.681746,0.090223,0.940640,195.406479,38863.476127,38864.416767
2,1,40000,0.681172,0.098994,0.717397,48.437195,9633.445976,9634.163374
0,1,10000,0.677645,0.062284,0.636658,9.562356,1901.812136,1902.448793


Running the zero shot TabPFN with chosen params. Same params are used for fine tuned variant.

In [ ]:
results = {}
layer_probas = {}

tabpfn_n_estimators = 2
tabpfn_subsample_samples = 20_000
predict_batch_size = 20_000

for name in tqdm(layer_files, desc="TabPFN layers"):
    key = f"{name}_subsampled"
    X_train, y_train, X_val, y_val, X_test, y_test = get_train_val_test(key)

    t0 = time.time()
    clf = TabPFNClassifier.create_default_for_version(
        ModelVersion.V3_5,
        device="cuda",
        n_estimators=tabpfn_n_estimators,
        inference_config={"SUBSAMPLE_SAMPLES": tabpfn_subsample_samples},
        random_state=42,
    )
    clf.fit(X_train, y_train)

    val_proba = predict_proba_chunked(clf, X_val, batch_size=predict_batch_size, desc="predicting val batches")
    threshold = tune_threshold(y_val, val_proba, beta=beta)

    y_proba = predict_proba_chunked(clf, X_test, batch_size=predict_batch_size, desc="predicting test batches")
    y_pred = (y_proba >= threshold).astype(int)
    elapsed = time.time() - t0

    metrics = {
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
        "mcc": matthews_corrcoef(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "threshold": threshold,
        "n_estimators": tabpfn_n_estimators,
        "subsample_samples": tabpfn_subsample_samples,
        "train_seconds": elapsed,
    }
    results[key] = metrics

    # checkpoint after every layer so a disconnect doesn't cost the whole run
    with open(results_dir / f"{key}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    np.save(results_dir / f"{key}_y_proba.npy", y_proba)
    np.save(results_dir / f"{key}_y_pred.npy", y_pred)

    # cache for beta sweep
    layer_probas[key] = {
        "y_val": y_val,
        "val_proba": val_proba,
        "y_test": y_test,
        "test_proba": y_proba,
    }

    print(f"{key}: {metrics}")

    del clf, X_train, y_train, X_val, X_test, y_pred
    gc.collect()
    torch.cuda.empty_cache()

TabPFN layers:   0%|          | 0/4 [00:00<?, ?it/s]

predicting val batches:   0%|          | 0/8 [00:00<?, ?it/s]

predicting test batches:   0%|          | 0/299 [00:00<?, ?it/s]

layer_a_bts_subsampled: {'f1': 0.4218636795925581, 'roc_auc': np.float64(0.6727551706421926), 'pr_auc': np.float64(0.3564161390627768), 'mcc': np.float64(0.2063496986535998), 'precision': 0.30938879153976867, 'recall': 0.6628272138858504, 'threshold': np.float64(0.1919916868209839), 'n_estimators': 2, 'subsample_samples': 20000, 'train_seconds': 5945.119601011276}


predicting val batches:   0%|          | 0/8 [00:00<?, ?it/s]

...

predicting test batches:   0%|          | 0/299 [00:00<?, ?it/s]

layer_c_bts_gdelt_subsampled: {'f1': 0.42372017817484825, 'roc_auc': np.float64(0.6744619615786263), 'pr_auc': np.float64(0.35719226222663963), 'mcc': np.float64(0.20867838725315152), 'precision': 0.3077946144026954, 'recall': 0.6797281188778038, 'threshold': np.float64(0.1919897049665451), 'n_estimators': 2, 'subsample_samples': 20000, 'train_seconds': 6567.3534960746765}


predicting val batches:   0%|          | 0/8 [00:00<?, ?it/s]

predicting test batches:   0%|          | 0/299 [00:00<?, ?it/s]

layer_d_bts_meteostat_gdelt_subsampled: {'f1': 0.4295361711361396, 'roc_auc': np.float64(0.6821080494544531), 'pr_auc': np.float64(0.3667382487062542), 'mcc': np.float64(0.21919367597588454), 'precision': 0.3153358344992071, 'recall': 0.6734174146900491, 'threshold': np.float64(0.18911001086235046), 'n_estimators': 2, 'subsample_samples': 20000, 'train_seconds': 6707.138389110565}


In [12]:
with open(results_dir / "all_layers_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

pd.DataFrame(results).T

,f1,roc_auc,pr_auc,mcc,precision,recall,threshold,n_estimators,subsample_samples,train_seconds
layer_a_bts_subsampled,0.421864,0.672755,0.356416,0.206350,0.309389,0.662827,0.191992,2.0,20000.0,5945.119601
layer_b_bts_meteostat_subsampled,0.428440,0.681979,0.366251,0.216020,0.305810,0.715261,0.176115,2.0,20000.0,6073.654695
layer_c_bts_gdelt_subsampled,0.423720,0.674462,0.357192,0.208678,0.307795,0.679728,0.191990,2.0,20000.0,6567.353496
layer_d_bts_meteostat_gdelt_subsampled,0.429536,0.682108,0.366738,0.219194,0.315336,0.673417,0.189110,2.0,20000.0,6707.138389


In [13]:
summary_df = pd.DataFrame(results).T
summary_df.index.name = "layer"
summary_df = summary_df.reset_index()

# match RF summary's column naming/order where possible
summary_df = summary_df.rename(columns={"roc_auc": "auc_roc", "pr_auc": "auc_pr", "train_seconds": "fit_time_s"})
summary_df = summary_df[["layer", "f1", "precision", "recall", "auc_roc", "auc_pr", "mcc", "threshold", "n_estimators", "subsample_samples", "fit_time_s"]]

summary_df.to_csv(results_dir / "fm_subsampled_summary.csv", index=False)
summary_df

,layer,f1,precision,recall,auc_roc,auc_pr,mcc,threshold,n_estimators,subsample_samples,fit_time_s
0,layer_a_bts_subsampled,0.421864,0.309389,0.662827,0.672755,0.356416,0.206350,0.191992,2.0,20000.0,5945.119601
1,layer_b_bts_meteostat_subsampled,0.428440,0.305810,0.715261,0.681979,0.366251,0.216020,0.176115,2.0,20000.0,6073.654695
2,layer_c_bts_gdelt_subsampled,0.423720,0.307795,0.679728,0.674462,0.357192,0.208678,0.191990,2.0,20000.0,6567.353496
3,layer_d_bts_meteostat_gdelt_subsampled,0.429536,0.315336,0.673417,0.682108,0.366738,0.219194,0.189110,2.0,20000.0,6707.138389


Run feature importance on small sample.

In [ ]:
from sklearn.inspection import permutation_importance

importance_sample_size = 3_000

importance_results = {}

for name in tqdm(layer_files, desc="Feature importance"):
    key = f"{name}_subsampled"
    X_train, y_train, X_val, y_val, X_test, y_test = get_train_val_test(key)

    #same config used in the main loop
    clf = TabPFNClassifier(
        device="cuda",
        n_estimators=tabpfn_n_estimators,
        inference_config={"SUBSAMPLE_SAMPLES": tabpfn_subsample_samples},
        random_state=42
    )
    clf.fit(X_train, y_train)

   
    sample_idx = X_test.sample(n=min(importance_sample_size, len(X_test)), random_state=42).index
    X_sample = X_test.loc[sample_idx]
    y_sample = y_test.loc[sample_idx]

    t0 = time.time()
    perm_result = permutation_importance(
        clf, X_sample, y_sample,
        scoring="roc_auc",
        n_repeats=5,
        random_state=42,
        n_jobs=1,
    )
    elapsed = time.time() - t0

    importance_df = pd.DataFrame({
        "feature": X_sample.columns,
        "importance_mean": perm_result.importances_mean,
        "importance_std": perm_result.importances_std,
    }).sort_values("importance_mean", ascending=False)

    importance_df.to_csv(results_dir / f"{key}_feature_importance.csv", index=False)
    importance_results[key] = importance_df

    print(f"{key}: computed in {elapsed:.1f}s")
    print(importance_df.head(10))

    del clf, X_train, y_train, X_val, X_test, X_sample, y_sample
    gc.collect()
    torch.cuda.empty_cache()

Feature importance:   0%|          | 0/4 [00:00<?, ?it/s]

layer_a_bts_subsampled: computed in 1456.8s
             feature  importance_mean  importance_std
6           hour_sin         0.105714        0.009647
3          month_cos         0.019874        0.004094
7           hour_cos         0.007294        0.001135
9            dest_te         0.005894        0.001643
0   crs_elapsed_time         0.005019        0.001936
8          origin_te         0.004921        0.000908
4            dow_sin         0.003970        0.001181
23        carrier_WN         0.002955        0.001404
1           distance         0.002689        0.000480
22        carrier_UA         0.002230        0.000125
layer_b_bts_meteostat_subsampled: computed in 1727.6s
             feature  importance_mean  importance_std
10          hour_sin         0.110840        0.009001
7          month_cos         0.013786        0.003918
2      temp_mean_12h         0.007887        0.003172
12         origin_te         0.005968        0.001291
13           dest_te         0.005221 